# Theoretical Framework: Depth-Integrated Free-Surface Flow Models

Based on the initial derivations from Section 2 through Section 2.2 of *Yang and Liu (2024)*, this framework delineates the transformation from an exact 3-D Euler description to the vertically mapped formulation, stopping prior to the introduction of horizontal spatial depth-averaging/discretization approximations.

In [1]:
# ============================================================
#  LFE-M Depth-Integrated Wave Solver
#  Arbitrary vertical polynomial degree p, M elements
#  Horizontal discretization: Gridap 
# ============================================================

# [Surface Layer]  ------ Node Ns+1  --> Unknowns: (u_Ns+1, v_Ns+1) AND Global Depth H
#                 |    
#                 | Element M (Layer M)
#                 |   
#                 ------ 
#                 |
#                 | Element M-1
#                 |
#                 ...
#                 |
#                 ------ 
#                 |
#                 | Element 1 (Bottom Layer)
#                 |
# [Seabed Layer]  ------ Node 0    --> Unknowns: (u_0, v_0)

using Gridap
using Gridap.FESpaces
using Gridap.ReferenceFEs
using Gridap.Geometry
using Gridap.Arrays
using Gridap.ODEs
using Gridap.CellData
using Gridap.ODEs
using LinearAlgebra

include("../src/TS_2HDmodel.jl")  # Import the module containing the solver implementation
using .TS_2HDmodel

LoadError: LoadError: ArgumentError: Package FillArrays not found in current path.
- Run `import Pkg; Pkg.add("FillArrays")` to install the FillArrays package.
in expression starting at /home/pmanyerfuertes/Documents/TU-Delft/WP2/SWE_farfield/TS_2HDmodel/src/TS_2HDmodel.jl:1

In [2]:
# Build vertical FEM approximation space

"""
    build_vertical_model(M, c_bdy)

Create a 1D Gridap CartesianDiscreteModel for sigma in [0,1].
M elements with boundaries at c_bdy (length M+1).
Uses a piecewise-linear coordinate map so each element has the correct
physical size; "tag_1" = sigma=0, "tag_2" = sigma=1.
"""
function build_vertical_model(M::Int, c_bdy::Vector{Float64})
    @assert length(c_bdy) == M + 1
    @assert c_bdy[1] ≈ 0.0 && c_bdy[end] ≈ 1.0

    function sigma_map(x)
        t = x[1]
        for k in 1:M
            a = (k-1) / M
            b = k / M
            if t <= b + 1e-14
                local_t = (t - a) / (b - a)
                return VectorValue(c_bdy[k] + local_t * (c_bdy[k+1] - c_bdy[k]))
            end
        end
        return VectorValue(c_bdy[end])
    end

    return CartesianDiscreteModel((0.0, 1.0), M; map=sigma_map)
end

"""
    build_vertical_fe_spaces(sigma_model, p)

Build trial/test FE spaces on the sigma mesh for:
    - the vertical FE approximation basis ϕ_j
    - the A/D integration unit vertical basis φ_j = ∫ ϕ_j dσ
        -> The problem φ_j = ∫_0^σ ϕ_j(σ') dσ' is equivalent to solving dφ_j/dσ = ϕ_j with φ_j(0)=0
        -> if ϕ_j is degree p, then φ_j is degree p+1

Hence, the generated FE spaces are:
    - for ϕ_j: degree p, no Dirichlet BCs:
        -> V_phi: test FE space
        -> U_phi: trial FE space
    - for φ_j: degree p+1, Dirichlet BC φ_j(0)=0 at sigma=0 ("tag_1")
        -> V_varphi: test FE space
        -> U_varphi: trial FE space (enforces φ_j(0)=0)
"""
function build_vertical_fe_spaces(sigma_model, p::Int)
    # Build FE spaces for vertical basis functions ϕ_j
    reffe_phi = ReferenceFE(lagrangian, Float64, p)
    V_phi = FESpace(sigma_model, reffe_phi; conformity=:H1)
    U_phi = TrialFESpace(V_phi)
    # Build FE spaces for unit vertical basis functions φ_j
    reffe_varphi   = ReferenceFE(lagrangian, Float64, p + 1)
    V_varphi   = FESpace(sigma_model, reffe_varphi;   conformity=:H1, dirichlet_tags=["tag_1"]) # sigma = 0 -> sea bottom no flux condition
    U_varphi   = TrialFESpace(V_varphi, 0.0)  # Dirichlet BC for vertical velocity at sigma=0 (bottom)

    return V_phi, U_phi, V_varphi, U_varphi
end


"""
    compute_unit_varphi(ϕ, V_varphi, U_varphi, dσ)

Solve for φ: unit vertical velocity FEFunction.

H1-seminorm projection: enforces dφj/dσ = ϕj with φ(0)=0.
  a(φj,v) = ∫ (dφj/dσ)(dv/dσ) dσ = ∫ ∇φj ⋅ ∇v dσ
  l(v)   = ∫ ϕj (dv/dσ) dσ = ∫ ϕj ∇v dσ

"""
function compute_unit_varphi(ϕj, V_varphi, U_varphi, dσ)
    # φ BVP LHS:
    a(φj, v) = ∫(∇(φj) ⋅ ∇(v) )*dσ
    # φ BVP RHS:
    l(v)    = ∫((∇(v) ⋅ VectorValue(1.0)) * ϕj)*dσ
    # Gridap operator and solve
    op      = AffineFEOperator(a, l, U_varphi, V_varphi)
    return solve(op)
end

g = 9.81        # Gravity acceleration
h(x) = 10.0     # Water depth function (constant for now)

M = 2           # Number of vertical elements
p = 1           # Vertical polynomial degree
degree = 2*p+2  # Vertical quadrature degree 

c_bdy   = [0.0, 0.728, 1.0]   # LFE-2 optimised node positions
sigma_model = build_vertical_model(M, c_bdy)

V_phi, U_phi, V_varphi, U_varphi = build_vertical_fe_spaces(sigma_model, p)
Ns = num_free_dofs(U_phi)

Ω_sig = Triangulation(sigma_model)
dσ = Measure(Ω_sig,degree)

GenericMeasure()

In [3]:
ϕvec = Vector{Any}(undef, Ns)  # Basis functions ϕ_j as FEFunctions 
dϕvec = Vector{Any}(undef, Ns) # Derivatives of basis functions dϕ_j/dσ as FEFunctions
φvec = Vector{Any}(undef, Ns)  # Unit vertical velocity φ_j as FEFunctions
Φvec = Vector{Any}(undef, Ns)  # Depth-average weights (scalars)
for j in 1:Ns
    e_j        = zeros(Float64, Ns)
    e_j[j]     = 1.0
    ϕvec[j] = FEFunction(U_phi, e_j)
    dϕvec[j] = ∇(ϕvec[j]) ⋅ VectorValue(1.0)   # dϕⱼ/dσ (scalar CellField)
    φvec[j] = compute_unit_varphi(ϕvec[j], V_varphi, U_varphi, dσ)
    Φvec[j] = sum(∫(ϕvec[j]) * dσ)  # function sum collapses the CellField contributions to a scalar
end

function θjfun(σ, j)
    return VectorValue(evaluate(ϕvec[j], σ),
                       σ[1] * evaluate(ϕvec[j], σ),
                       evaluate(φvec[j], σ))
end

θjfun (generic function with 1 method)

In [4]:
function θjcellfield(σ::CellField, j::Int)
    return Operation(VectorValue)(ϕvec[j],
                                  σ * ϕvec[j],
                                  φvec[j])
end

function Θkjcellfield(σ::CellField, k::Int, j::Int)
    return Operation(VectorValue)(σ * Φvec[k] * ϕvec[j],
                                  Φvec[k] * ϕvec[j],
                                  ϕvec[j] * ϕvec[k],
                                  σ * ϕvec[j] * ϕvec[k],
                                  φvec[j]* ϕvec[k],
                                  σ * Φvec[j] * dϕvec[k] - φvec[j] * dϕvec[k],
                                  σ * Φvec[j] * ϕvec[k] + σ*σ * Φvec[j] * dϕvec[k] - φvec[j] * dϕvec[k] - σ * φvec[j] * dϕvec[k],
                                  σ * Φvec[j] * ϕvec[k] - φvec[j] * ϕvec[k])
end



# Build vertical tensors

M2 = zeros(Float64, Ns, Ns)                      # Vertical mass 2nd order tensor 
M3 = zeros(Float64, Ns, Ns, Ns)                  # Vertical mass 3rd order tensor
G3 = zeros(Float64, Ns, Ns, Ns)                  # Vertical advection gradient 3rd order tensor 
A2 = zeros(VectorValue{3, Float64}, Ns, Ns)      # Vertical linear pressure
K2 = zeros(VectorValue{3, Float64}, Ns, Ns)      # Vertical linear pressure gradient
A3 = zeros(VectorValue{8, Float64}, Ns, Ns, Ns)  # Vertical non-linear pressure 3rd order tensor A
K3 = zeros(VectorValue{8, Float64}, Ns, Ns, Ns)  # Vertical non-linear pressure 3rd order tensor K

# Define CellField returning the value of the evaluated sigma inside the integral -> placeholder for quadrature point coordinate
σ = CellField(x -> x[1], Ω_sig)

for i in 1:Ns
    for j in 1:Ns
        
        M2[i,j] = sum(∫(ϕvec[i] * ϕvec[j]) * dσ)
        A2[i,j] = sum(∫(ϕvec[i] * θjcellfield(σ, j)) * dσ)
        K2[i,j] = sum(∫(θjcellfield(σ, j) * ( φvec[i] - σ * ϕvec[i] ) ) * dσ)
        
        for k in 1:Ns
            M3[i,j,k] = sum(∫(ϕvec[i] * ϕvec[j] * ϕvec[k]) * dσ)
            G3[i,j,k] = sum(∫((σ * Φvec[k] - φvec[k]) * dϕvec[j] * ϕvec[i]) * dσ)
            K3[i,j,k] = sum(∫(Θkjcellfield(σ, k, j) * ( φvec[i] - σ * ϕvec[i] ) ) * dσ)
        end
    end
end

In [5]:
##### Build horizontal FE spaces 
# Horizontal domain 
domain = (0.0, 10.0, 0.0, 10.0)
N  = (20, 20)
model = CartesianDiscreteModel(domain, N)

# Horizontal quadrature
degree = 4
Ωₕ = Triangulation(model)
dΩ = Measure(Ωₕ,degree)

# Water depth FE space: Float64 for H
# No Dirichlet BCs for H
orderH = 2
reffeH = ReferenceFE(lagrangian, Float64, orderH)
testFE_H = FESpace(model, reffeH; conformity=:H1)
trialFE_H = TrialFESpace(testFE_H)

# Horizontal velocity FE space: VectorValue{2, Float64} for (u,v) horizontal velocity components
# Dirichlet BCs??????
orderU = 2 
reffeU = ReferenceFE(lagrangian, VectorValue{2, Float64}, orderU)
MultilayerTestFE_U = [FESpace(model, reffeU; conformity=:H1) for i in 1:Ns]
MultilayerTrialFE_U = [TrialFESpace(testFE_U) for testFE_U in MultilayerTestFE_U]

# Block multifield FE spaces for the 2HD model
MultilayerTestFE = MultiFieldFESpace([testFE_H, MultilayerTestFE_U...])
MultilayerTrialFE = MultiFieldFESpace([trialFE_H, MultilayerTrialFE_U...])

MultiFieldFESpace()

In [6]:
##### Residual 

# ----------------------------------------------------------
#  CellField linear combination helpers
# ----------------------------------------------------------

"""Σⱼ cⱼ * fieldⱼ  — explicit accumulation, no CellField closure."""
function field_sum(coeffs::Vector{Float64}, fields::Vector)
    acc = coeffs[1] * fields[1]
    for j in 2:length(coeffs)
        acc = acc + coeffs[j] * fields[j]
    end
    return acc
end

"""Σⱼ M[i,j] * fieldⱼ  — row-i matrix-vector product."""
function mat_field_sum(Mmat::Matrix{Float64}, i::Int, fields::Vector)
    acc = Mmat[i,1] * fields[1]
    for j in 2:size(Mmat,2)
        acc = acc + Mmat[i,j] * fields[j]
    end
    return acc
end

"""Σⱼ M[i,j] * ∇(fieldⱼ)  — row-i weighted gradient sum (returns VectorValue CellField)."""
function mat_gradvec_sum(Mmat::Matrix{Float64}, i::Int, fields::Vector)
    acc = Mmat[i,1] * ∇(fields[1])
    for j in 2:size(Mmat,2)
        acc = acc + Mmat[i,j] * ∇(fields[j])
    end
    return acc
end


function residual(t,U,V)
    # U = [H, u1, u2, ..., u_Ns]         # Multifield solution vector
    # V = [q, v_u1, v_u2, ..., v_uNs]  # Multifield test function vector

    # Time derivative of the solution vector U
    Ut = ∂t(U) 

    H = U[1]         # Water depth FE test field
    Ht = Ut[1]       # Time derivative of water depth FE test field
    uj = U[2:end]    # Horizontal velocity FE test fields for each layer
    utj = Ut[2:end]  # Time derivative of horizontal velocity FE test fields for each layer
    q = V[1]         # Water depth FE trial function
    vj = V[2:end]    # Horizontal velocity FE trial fields for each layer

    h = CellField(h_func, Ωₕ) # Barimetry
    

    # Mass Continuity residual: 
    res_mass = ∫( q*Ht - ∇(q) ⋅ (H*uj) ⋅ Φvec ) * dΩ

    # Horizontal Momentum residual:
    # Acceleration term: 
    res_acc = ∫( H * (M2 * utj) ⋅ vj ) * dΩ  

    # Advection term:
    FM = (∇(uj)') * M3 * uj 
    FG = (∇(H * uj)') * G3 * uj 
    
    res_advec =∫( (H * F_M + F_G) ⋅ vj ) * dΩ

    # Gravity term:
    res_grav = ∫( g * H * ∇(H-h) * Φvec ⋅ vj ) * dΩ

    # Linear pressure term:
    function L(Ut, H, h)
        return Operation(VectorValue)( - (∇(h)') ⋅ Ut, 
                                      (∇(H)') ⋅ Ut, 
                                      - ∇⋅(H*Ut))
    end
    res_linpress = ∫( H * ( ∇(h) * (L(ujt, H, h) ⊙ A2) + ∇(H) * (L(ujt, H, h) ⊙ K2) ) ⋅ vj ) * dΩ

    # Non-linear pressure term:
    function N(U, H, h)
        return Operation(VectorValue)( - (∇(∇⋅(H*U))') ⋅ U,
                                       ∇⋅((∇⋅(H*U) *U)),
                                       - (∇((∇(h)')⋅U)') ⋅ U,
                                       (∇((∇(H)')⋅U)') ⋅ U,
                                       - (∇(∇⋅(H*U))') ⋅ U,
                                       -(∇⋅(H*U) * (∇(h)' ⋅ U))/H ,
                                       (∇⋅(H*U) * (∇(H)' ⋅ U))/H,
                                       -(∇⋅(H*U) * (∇⋅(H*U))/H ))
    end
    res_nonlinpress = ∫( H * ( ∇(h) * (N(ujt, H, h) ⊙ A3) + ∇(H) * (N(ujt, H, h) ⊙ K3) ) ⋅ vj ) * dΩ

    return res_mass + res_acc + res_advec + res_grav - res_linpress - res_nonlinpress
end

residual (generic function with 1 method)

In [7]:
op = TransientFEOperator(residual, MultilayerTestFE, MultilayerTrialFE)

TransientFEOpFromWeakForm()

In [8]:
nl_iter = 20
nl_tol = 1e-6
lin_solver = LUSolver()
nl_solver = NLSolver(lin_solver, method=:newton, iterations=nl_iter, ftol=nl_tol, show_trace=false)

Δt = 0.05
#θ = 0.5
#solver = ThetaMethod(nl_solver, Δt, θ)
solver = RungeKutta(nl_solver, lin_solver, Δt, :SDIRK_3_3)

t0, tF = 0.0, 1.0
U0 = FEFunction(MultilayerTestFE, zeros(Float64, num_free_dofs(MultilayerTestFE)))

Uh = solve(solver, op, t0, tF, U0)

for (t_n, u_n) in Uh
    println("t = $t_n, ||u|| = $(norm(u_n))")
end

MethodError: MethodError: no method matching lastindex(::TransientMultiFieldCellField{Gridap.MultiField.MultiFieldFEFunction{Gridap.MultiField.MultiFieldCellField{ReferenceDomain}}})

Closest candidates are:
  lastindex(::Any, !Matched::Any)
   @ Base abstractarray.jl:427
  lastindex(!Matched::Base.JuliaSyntax.SourceFile)
   @ Base /cache/build/builder-amdci5-7/julialang/julia-release-1-dot-10/base/JuliaSyntax/src/source_files.jl:132
  lastindex(!Matched::Base64.Buffer)
   @ Base64 ~/.julia/juliaup/julia-1.10.10+0.x64.linux.gnu/share/julia/stdlib/v1.10/Base64/src/buffer.jl:19
  ...
